# 结构化输出
结构化输出允许Agent按照特定的、可预测的格式（Schema）返回数据。无需再写代码来解析自然语言，可以直接拿到程序可以直接使用的json对象、Pydantic模型或dataclass形式的结构化数据

Langchain的create_agent可以自动处理结构化输出。用户设置其期望的结构化输出模式，当模型生成结构化数据时，它会被捕获、验证，并以 'structured_response' 键的形式返回到agent的state中。

```python
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],      #通过Tools来
        ProviderStrategy[StructuredResponseT],  #原生支持
        type[StructuredResponseT],
        None,
    ]
```

# Response format 响应格式
使用 `response_format` 来指定Agent要返回的结构
- ToolStrategy[StructuredResponseT] : 使用工具调用以实现结构化输出
- ProviderStrategy[StructuredResponseT] : 使用提供者原生的结构化输出
- type[StructuredResponseT] : 模式类型 - 根据模型能力自动选择最佳策略
- None : 未明确请求结构化输出

例如 OpenAI、Anthropic (Claude) 或 xAI (Grok)）就有原生支持结构化输出

结构化响应返回在Agent的最终状态中的 **structured_response** 键。final state

# ProviderStrategy 提供商策略
一些模型提供者通过他们的 API 原生支持结构化输出（例如 OpenAI、xAI（Grok）、Gemini、Anthropic（Claude））。当可用时，这是 **最可靠** 的方法。

使用这个要配置一个 ProviderStrategy 类

In [8]:
# from typing import Generic

# class ProviderStrategy(Generic[SchemaT]):
#     schema: type[SchemaT]       # 必选，Schema的格式
#     strict: bool | None = None # 是否严格遵循Schema格式，可选

要定义结构化输出的Schema有几种方式：
- Pydantic的BaseModel子类，具有字段验证，验证后返回Pydantic实例
- dataclass类，带类型注解的Python数据类，返回dict
- TypedDict子类，返回dict
- JSON Schema，JSON格式的dict，返回dict



In [1]:
from langchain_openai import ChatOpenAI
import os 

model = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    #! 这里要注意，模型的Tools能力也挂钩
    base_url="https://api.siliconflow.cn/v1",
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    temperature=0.2,
)

In [14]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


In [15]:

#? 使用Pydantic模型
class ContactInfo(BaseModel):
    """联系人信息"""
    name: str = Field(description="人名")
    email: str = Field(description="邮箱地址")
    phone: str = Field(description="电话号码")

In [16]:
agent = create_agent(
    model,
    response_format=ContactInfo  #? 这里会自动判断是否原生支持，还是工具来实现
)

res = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(res["structured_response"]) # 获取结构化输出，放在 "structured_response"

IndexError: list index out of range

# Tool calling strategy
对于原生不支持结构化输出的model，Langchain提供通过调用工具的方式来达到这一效果

不过要配置一个 `ToolStrategy` 

```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]               # 必选参数，支持的类型也是那几个
    tool_message_content: str | None    # 生成结构化输出时，工具消息的自定义内容。如果未提供，则默认显示显示结构化响应数据的消息
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal # 指定字面量
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy #!

class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="对于产品的评分", ge=1, le=5) #  1 <= x <= 5
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")
    


In [ ]:
agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(ProductReview) # 指定使用工具实现
)

result = agent.invoke({
    "messages": [{
        "role": "user", 
        "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"
        }]
})
result["structured_response"]

# 自定义工具消息内容
ToolStrategy 中的tool_message_content 可以自定义在生成结构化输出时，对话历史中显示的消息

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

toolStrategy = ToolStrategy(
    schema=ProductReview,
    tool_message_content="hhhhh"
)

agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(       #? 这玩意也可以在外面定义
        schema=MeetingAction, # 可以指定多种格式，然后会自动选择最合适的 Union
        tool_message_content="666结构化输出牛逼"
    )
)

res = agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

res["messages"][-1].pretty_print()

================================= Tool Message =================================
Name: MeetingAction

666结构化输出牛逼


# 错误处理
大模型在通过工具调用生成结构化信息的时候可以回出错，Langchain中有智能重试机制来自动处理这个错误

## 多个结构化输出 错误
当使用Union联合类型定义多个结构化Schema的时候，模型同时返回多个结构化响应，而系统只期望返回一个，从而发生错误

发生错误的时候会生成一个ToolMessage告诉模型出错了，要在思考一下。

当 schema 是 Union 联合类型时，模型需要做"二选一"的决策。但大语言模型有时会倾向于"给更多信息"，同时调用多个工具。这个错误处理机制就像一个"纠错老师"，告诉模型"你选多了，再想想"。


In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails], handle_errors=True)  # Default: handle_errors=True
)

res = agent.invoke({
    "messages": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"
})


IndexError: list index out of range

# 错误处理策略
返回一条str消息告诉LLM出错了，可以在 handle_errors 参数自定义错误处理方式

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)

也可以只处理特定的错误类型

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
) # 其他的异常就抛出

# 也可以处理多个，用元组就行
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Retry on ValueError and TypeError
)

# 还可以传入自定义函数
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"
    
ToolStrategy(
    schema=Union[ContactInfo, EventDetails],
    handle_errors=custom_error_handler
)  # Default: handle_errors=True

